# Notebook 10 — Integrated Coastal Groundwater-Subsidence Risk Index (CGSRI)
**Stage 10 of the Ghana Groundwater–Subsidence Research Pipeline**

Constructs the Coastal Groundwater-Subsidence Risk Index by combining hazard, exposure, and vulnerability components at the 10 km analysis grid. Handles the GRACE land-mask gap in western Ghana by substituting CHIRPS rainfall stress where GRACE-derived indicators are unavailable. Produces risk maps, municipality rankings, and priority intervention zones.

In [ ]:
import os
import pathlib; os.chdir(str(pathlib.Path.cwd().parent) if pathlib.Path.cwd().name == 'notebooks' else str(pathlib.Path.cwd()))

import warnings; warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import geopandas as gpd
import xarray as xr
import rasterio
from rasterio.mask import mask as rio_mask
from rasterio.transform import rowcol
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from matplotlib.patches import Patch
from pathlib import Path
from scipy.spatial import cKDTree

ROOT     = Path('.')
PROC_IDX = ROOT / 'data' / 'processed' / 'pressure_indices'
PROC_GW  = ROOT / 'data' / 'processed' / 'groundwater_anomalies'
PROC_SUB = ROOT / 'data' / 'processed' / 'subsidence'
PROC_RISK = ROOT / 'data' / 'processed' / 'risk_index'
FIG_DIR  = ROOT / 'outputs' / 'figures'
TAB_DIR  = ROOT / 'outputs' / 'tables'
POP_DIR  = ROOT / 'data' / 'interim' / 'harmonized_grids' / 'population'
BOUND_DIR = ROOT / 'data' / 'raw' / 'boundaries'

PROC_RISK.mkdir(parents=True, exist_ok=True)

TARGET_CRS = 'EPSG:32630'
print('Setup OK')


In [ ]:
# 2. Load UGPI grid and add GRACE coverage flag
grid = gpd.read_file(PROC_IDX / 'ugpi_grid10km.gpkg')
print(f'Grid cells: {len(grid)}')

# Flag cells that have valid GRACE-derived drought indicator
grid['has_grace'] = grid['drought_freq'].notna().astype(int)
print(f'Cells with GRACE coverage: {grid["has_grace"].sum()} / {len(grid)}')
print(f'Cells without GRACE coverage (western Ghana): {(grid["has_grace"]==0).sum()}')

# Note: GRACE land-mask covers only eastern Ghana (lon 0.25-1.25)
# Accra, Cape Coast, Sekondi-Takoradi fall in the uncovered western zone
print()
print('NOTE: GRACE land-mask covers eastern Ghana only (lon ~0.25-1.25E).')
print('      Climate stress for western cells will use CHIRPS rainfall anomaly.')


In [ ]:
# 3. CHIRPS spatially-varying climate stress (per 10km cell)
# Uses pre-computed below-normal fraction from the full CHIRPS 0.05° grid
# This replaces the previous uniform substitute and captures the north-south
# rainfall gradient properly across the study area.

chirps_spatial = pd.read_csv(
    ROOT / 'data' / 'processed' / 'climate_indices' / 'chirps_drought_stress_per_cell.csv'
)
# Merge on cell_id
grid = grid.merge(chirps_spatial[['cell_id','chirps_below_normal_pct','chirps_mean_rain_mm']],
                  on='cell_id', how='left')

# For cells WITH GRACE: combine GRACE drought_freq + CHIRPS below-normal
# For cells WITHOUT GRACE: use CHIRPS only
grace_stress = (grid['drought_freq'] - grid['drought_freq'].min()) / \
               (grid['drought_freq'].max() - grid['drought_freq'].min() + 1e-9)
chirps_stress = (grid['chirps_below_normal_pct'] - grid['chirps_below_normal_pct'].min()) / \
                (grid['chirps_below_normal_pct'].max() - grid['chirps_below_normal_pct'].min() + 1e-9)

grid['climate_stress'] = np.where(
    grid['has_grace'] == 1,
    0.5 * grace_stress.fillna(0) + 0.5 * chirps_stress,  # blend for GRACE cells
    chirps_stress                                          # CHIRPS only elsewhere
)
print(f'Climate stress (spatial CHIRPS): mean={grid["climate_stress"].mean():.3f}, '
      f'range=[{grid["climate_stress"].min():.3f}, {grid["climate_stress"].max():.3f}]')
print(f'CHIRPS below-normal % range: {grid["chirps_below_normal_pct"].min():.1f}% – {grid["chirps_below_normal_pct"].max():.1f}%')
print(f'Mean rain range: {grid["chirps_mean_rain_mm"].min():.0f} – {grid["chirps_mean_rain_mm"].max():.0f} mm/yr')


In [ ]:
# 4. GW storage decline hazard
# For GRACE cells: use normalised declining GWSA trend
# For non-GRACE cells: use ugpi_urban_only as GW demand proxy (higher demand = higher hazard)
gwsa_trend_vals = grid['gwsa_decline'].fillna(0)
slope_min = gwsa_trend_vals.min()
slope_max = gwsa_trend_vals.max()
gw_decline_norm = (gwsa_trend_vals - slope_min) / (slope_max - slope_min + 1e-9)

# For cells with no GRACE data, use normalised urban pressure as GW demand proxy
urban_proxy_norm = (grid['ugpi_urban_only'] - grid['ugpi_urban_only'].min()) / \
                   (grid['ugpi_urban_only'].max() - grid['ugpi_urban_only'].min() + 1e-9)

grid['gw_hazard'] = np.where(
    grid['has_grace'] == 1,
    gw_decline_norm,
    urban_proxy_norm * 0.5  # conservative 50% weight for proxy
)
print(f'GW hazard: mean={grid["gw_hazard"].mean():.3f}')


In [ ]:
# 5. Build subsidence hazard — blending proxy with MintPy SBAS InSAR velocity
# ------------------------------------------------------------------
# Proxy components: low elevation + high building density + high GW demand
# InSAR component:  MintPy SBAS LOS velocity (mm/yr); negative = moving away
#                   from satellite (subsidence-sense for ascending orbit)
# Sampling: only 113 of 1259 cells intersect the Sentinel-1 track-147 footprint
# (roughly Takoradi–Accra corridor). For these cells, use coverage-weighted
# blend. For cells outside the footprint, use proxy only.
# Caution: LOS ≠ vertical; ERA5 tropospheric correction applied (pyAPS3).
# Coastal zone (5.0–7.0°N) shows near-uniform valid pixel distribution,
# suggesting possible SBAS noise fitting. Blend weight caps at 0.70 for
# cells below 7.0°N as conservative approach (see Section 10.2 of methodology.md).
# ------------------------------------------------------------------

import warnings
from shapely.geometry import box as _box

dem_path = ROOT / 'data' / 'interim' / 'harmonized_grids' / 'dem_coastal50km_utm30n.tif'
vel_path = PROC_SUB / 'mintpy_velocity_los_era5.tif'  # ERA5 tropospheric correction applied

# ─── (a) Elevation component (proxy) ───────────────────────────────────────
elev_vals = np.full(len(grid), np.nan)
if dem_path.exists():
    grid_utm = grid.to_crs(TARGET_CRS)
    with rasterio.open(dem_path) as src:
        for i, geom in enumerate(grid_utm.geometry):
            try:
                out_img, _ = rio_mask(src, [geom.__geo_interface__], crop=True)
                vals = out_img[0].astype(float)
                nd_v = src.nodata
                if nd_v is not None: vals[vals == nd_v] = np.nan
                valid = vals[~np.isnan(vals)]
                elev_vals[i] = valid.mean() if len(valid) > 0 else np.nan
            except Exception: pass
    print(f'Elevation extracted: mean={np.nanmean(elev_vals):.1f} m, '
          f'range=[{np.nanmin(elev_vals):.0f}, {np.nanmax(elev_vals):.0f}] m')
else:
    print('DEM not found; flat elevation proxy used')

grid['elev_m'] = elev_vals
grid['elev_m'] = grid['elev_m'].fillna(grid['elev_m'].median())

elev_clipped = grid['elev_m'].clip(0, 200)
elev_inv = 1 - (elev_clipped - elev_clipped.min()) / \
               (elev_clipped.max() - elev_clipped.min() + 1e-9)
bd_norm  = (grid['builtup_density_2020'] - grid['builtup_density_2020'].min()) / \
           (grid['builtup_density_2020'].max() - grid['builtup_density_2020'].min() + 1e-9)
sub_proxy = (elev_inv + bd_norm + grid['gw_hazard']) / 3

# ─── (b) MintPy InSAR velocity component ───────────────────────────────────
insar_mean_vel = np.full(len(grid), np.nan)
insar_n_valid  = np.zeros(len(grid), dtype=int)   # absolute valid pixel count

if vel_path.exists():
    with rasterio.open(vel_path) as vsrc:
        nodata_v = vsrc.nodata if vsrc.nodata is not None else -9999.0

        # Pre-filter: only process cells that intersect the InSAR raster extent
        insar_box  = _box(*vsrc.bounds)
        grid_wgs84 = grid.to_crs('EPSG:4326')
        in_footprint = grid_wgs84.intersects(insar_box)
        n_fp = int(in_footprint.sum())
        print(f'Cells in InSAR footprint: {n_fp} / {len(grid)}')

        for i in np.where(in_footprint)[0]:
            geom = grid_wgs84.geometry.iloc[i]
            try:
                out_img, _ = rio_mask(vsrc, [geom.__geo_interface__], crop=True,
                                      all_touched=True)
                v = out_img[0].astype(float)
                v[v == nodata_v]   = np.nan
                v[np.abs(v) > 300] = np.nan  # sanity mask (>300 mm/yr is noise)
                valid_px = int(np.sum(~np.isnan(v)))
                insar_n_valid[i]  = valid_px
                if valid_px >= 10:  # at least 10 valid pixels (~0.06 km² coverage)
                    insar_mean_vel[i] = float(np.nanmean(v))
            except Exception:
                pass

    n_usable = int(np.sum(insar_n_valid >= 10))
    print(f'Cells with ≥10 valid InSAR pixels: {n_usable} / {n_fp} (in-footprint)')
    if n_usable > 0:
        valid_vels = insar_mean_vel[insar_n_valid >= 10]
        print(f'Mean LOS velocity (usable cells): {np.nanmean(valid_vels):.1f} mm/yr '
              f'(range: {np.nanmin(valid_vels):.1f} to {np.nanmax(valid_vels):.1f})')
        print(f'  Median valid pixels per covered cell: '
              f'{np.median(insar_n_valid[insar_n_valid > 0]):.0f}')
else:
    print('MintPy velocity not found — using proxy only')

grid['insar_vel_mmyr'] = insar_mean_vel
grid['insar_n_valid']  = insar_n_valid

# ─── (c) Normalise InSAR to hazard indicator (0 = uplift, 1 = subsidence) ─
# Clip velocity to [-60, +20] mm/yr to reduce atmospheric outlier influence.
# Invert: very negative LOS → high subsidence hazard score.
HAS_INSAR = insar_n_valid >= 10
vel_valid  = insar_mean_vel[HAS_INSAR]
if len(vel_valid) > 0:
    vel_clipped_valid = np.clip(vel_valid, -60, 20)
    vel_min = float(np.nanmin(vel_clipped_valid))
    vel_max = float(np.nanmax(vel_clipped_valid))
else:
    vel_min, vel_max = -30, 10  # fallback range

insar_hazard = np.full(len(grid), np.nan)
for i in np.where(HAS_INSAR)[0]:
    v_c = np.clip(insar_mean_vel[i], -60, 20)
    # Score: 1 = very negative (subsiding), 0 = positive (uplifting)
    insar_hazard[i] = 1.0 - (v_c - vel_min) / (vel_max - vel_min + 1e-9)

# ─── (d) Coverage-weighted blend: proxy × (1−w) + InSAR × w ───────────────
# Weight scales from 0 (< 10 px) to 1.0 (≥ 500 px).
# This keeps proxy as baseline; InSAR contribution grows with data density.
# Cap InSAR weight at 0.70 for coastal cells (lat < 7.0°N) — SBAS noise caution
insar_w = np.clip(insar_n_valid / 500.0, 0.0, 1.0)
coastal_mask = (grid['centroid_lat'] < 7.0).values
insar_w = np.where(coastal_mask, np.minimum(insar_w, 0.70), insar_w)
insar_w[~HAS_INSAR] = 0.0   # cells with < 10 pixels get full proxy weight

insar_hazard_filled = np.where(np.isnan(insar_hazard), sub_proxy, insar_hazard)
grid['sub_hazard'] = (1.0 - insar_w) * sub_proxy + insar_w * insar_hazard_filled

pct_insar = (insar_w > 0).mean() * 100
print(f'\nSub_hazard (blended): mean={grid["sub_hazard"].mean():.3f}')
print(f'  Cells using InSAR blend: {int((insar_w > 0).sum())} ({pct_insar:.1f}%)')
print(f'  Cells proxy-only:        {int((insar_w == 0).sum())} ({100-pct_insar:.1f}%)')
print(f'  Caution: InSAR LOS velocity is not vertical displacement;')
print(f'  atmospheric and seasonal signals not fully corrected.')


In [ ]:
# 6. Exposure component — population + infrastructure density

# Population density (normalised)
pop_norm = (grid['pop_density_km2'] - grid['pop_density_km2'].min()) / \
           (grid['pop_density_km2'].max() - grid['pop_density_km2'].min() + 1e-9)

# Building density (normalised) — proxy for infrastructure exposure
bd_norm_exp = (grid['builtup_density_2020'] - grid['builtup_density_2020'].min()) / \
              (grid['builtup_density_2020'].max() - grid['builtup_density_2020'].min() + 1e-9)

# Population growth rate — growing populations increase future exposure
pgr_norm = (grid['pop_growth_rate'] - grid['pop_growth_rate'].min()) / \
           (grid['pop_growth_rate'].max() - grid['pop_growth_rate'].min() + 1e-9)

grid['exposure'] = (pop_norm + bd_norm_exp + pgr_norm) / 3
print(f'Exposure: mean={grid["exposure"].mean():.3f}, max={grid["exposure"].max():.3f}')


In [ ]:
# 7. Vulnerability component — piped water access + distance to coast

# Piped water access proxy (from UGPI)
pwa_norm = grid['pwa_proxy']  # already 0-1

# Distance to coast proxy: Ghana coast is at ~4.5-5.5°N latitude
# Lower latitude = closer to coast = more vulnerable
COAST_LAT = 4.8  # approximate mean coastal latitude
grid['dist_coast_km'] = (grid['centroid_lat'] - COAST_LAT).clip(lower=0) * 111.0
dc_max = grid['dist_coast_km'].max()
coast_vuln = 1 - (grid['dist_coast_km'] / dc_max)

# Built-up growth rate — rapid growth = less adaptive capacity
bugr_norm = (grid['builtup_growth_pct'] - grid['builtup_growth_pct'].min()) / \
            (grid['builtup_growth_pct'].max() - grid['builtup_growth_pct'].min() + 1e-9)

grid['vulnerability'] = (pwa_norm + coast_vuln + bugr_norm) / 3
print(f'Vulnerability: mean={grid["vulnerability"].mean():.3f}, max={grid["vulnerability"].max():.3f}')


In [ ]:
# 8. Compute Coastal Groundwater-Subsidence Risk Index
# CGSRI = α*GW_Hazard + β*Sub_Hazard + γ*Climate_Stress + ε*Exposure + ζ*Vulnerability
# Equal weights as baseline

CGSRI_COMPONENTS = {
    'gw_hazard':     ('GW storage hazard',      0.20),
    'sub_hazard':    ('Subsidence hazard',       0.20),
    'climate_stress':('Climate stress',          0.20),
    'exposure':      ('Population/infra exposure',0.20),
    'vulnerability': ('Vulnerability',           0.20),
}

grid['cgsri'] = sum(
    w * grid[comp]
    for comp, (label, w) in CGSRI_COMPONENTS.items()
)

# Classify into 5 risk tiers using natural breaks (Jenks-like quantile approach)
# Use quintiles for transparent, reproducible classification
q20 = grid['cgsri'].quantile(0.20)
q40 = grid['cgsri'].quantile(0.40)
q60 = grid['cgsri'].quantile(0.60)
q80 = grid['cgsri'].quantile(0.80)

def classify_risk(val):
    if pd.isna(val): return 'Unknown'
    if val <= q20:   return 'Very low'
    if val <= q40:   return 'Low'
    if val <= q60:   return 'Moderate'
    if val <= q80:   return 'High'
    return 'Very high'

grid['cgsri_class'] = grid['cgsri'].apply(classify_risk)

print(f'CGSRI range: {grid["cgsri"].min():.3f} – {grid["cgsri"].max():.3f}')
print(f'Mean: {grid["cgsri"].mean():.3f}  |  Quintile thresholds: {q20:.3f}, {q40:.3f}, {q60:.3f}, {q80:.3f}')
print()
print('Risk class distribution (quintile thresholds):')
order = ['Very low','Low','Moderate','High','Very high']
for cls in order:
    cnt = (grid['cgsri_class'] == cls).sum()
    pct = cnt / len(grid) * 100
    bar = '█' * int(pct / 2)
    print(f'  {cls:12s}: {cnt:4d} ({pct:4.1f}%) {bar}')


In [ ]:
# 9. Municipality / district risk ranking
districts = gpd.read_file(BOUND_DIR / 'coastal_districts.gpkg').to_crs(TARGET_CRS)
print(f'Coastal districts: {len(districts)}')
print(f'District columns: {[c for c in districts.columns if c != "geometry"]}')

# Spatial join: assign each 10km cell to a district
grid_centroids = grid.to_crs(TARGET_CRS).copy()
grid_centroids['geometry'] = grid_centroids.geometry.centroid

joined = gpd.sjoin(grid_centroids, districts, how='left', predicate='within')

# Find district name column
name_col = next((c for c in districts.columns
                 if any(k in c.lower() for k in ['name','district','adm'])
                 and c != 'geometry'), districts.columns[0])
print(f'Using district name column: {name_col}')

district_stats = joined.groupby(name_col).agg(
    n_cells=('cgsri','count'),
    cgsri_mean=('cgsri','mean'),
    cgsri_max=('cgsri','max'),
    cgsri_p75=('cgsri', lambda x: x.quantile(0.75)),
    pop_total=('population_2020','sum'),
    builtup_km2_2025=('builtup_m2_2025', lambda x: x.sum()/1e6),
    high_risk_cells=('cgsri_class', lambda x: (x.isin(['High','Very high'])).sum()),
).round(3).sort_values('cgsri_mean', ascending=False).reset_index()

print(f'\nTop 15 districts by mean CGSRI:')
print(district_stats.head(15).to_string(index=False))

district_stats.to_csv(TAB_DIR / 'cgsri_district_ranking.csv', index=False)
print(f'\nSaved district ranking: {len(district_stats)} districts')


In [ ]:
# 10. Priority intervention zones
# Cells in top 10% CGSRI with high population (top 50% pop density)
cgsri_p90  = grid['cgsri'].quantile(0.90)
pop_median = grid['pop_density_km2'].median()

priority = grid[
    (grid['cgsri'] >= cgsri_p90) &
    (grid['pop_density_km2'] >= pop_median)
].copy()
print(f'Priority intervention zones (top 10% CGSRI + above-median population): {len(priority)} cells')
print(f'  Population in priority zones: {priority["population_2020"].sum():,.0f}')
print(f'  Built-up area 2025: {priority["builtup_m2_2025"].sum()/1e6:.1f} km²')
print()

# Top cells by location
top_priority = priority.nlargest(20, 'cgsri')[
    ['cell_id','centroid_lat','centroid_lon','cgsri','cgsri_class',
     'population_2020','builtup_density_2020','sub_hazard','exposure']
].reset_index(drop=True)
print('Top 20 priority cells:')
print(top_priority[['centroid_lat','centroid_lon','cgsri','cgsri_class',
                     'population_2020']].to_string(index=False))

top_priority.to_csv(TAB_DIR / 'cgsri_priority_zones.csv', index=False)


In [ ]:
# 11. Save CGSRI grid
cgsri_cols = [
    'cell_id','centroid_lat','centroid_lon',
    'gw_hazard','sub_hazard','climate_stress','exposure','vulnerability',
    'cgsri','cgsri_class','has_grace',
    'population_2020','pop_density_km2','builtup_density_2020',
    'builtup_m2_2025','elev_m','dist_coast_km'
]
grid[cgsri_cols].to_csv(PROC_RISK / 'cgsri_grid10km.csv', index=False)
grid.to_file(PROC_RISK / 'cgsri_grid10km.gpkg', driver='GPKG')
print('Saved CGSRI grid CSV and GeoPackage')


In [ ]:
# 12. Figure: CGSRI component maps (3x2 panel)
import cartopy.crs as ccrs
import cartopy.feature as cfeature

components = [
    ('gw_hazard',      'A  GW Storage Hazard'),
    ('sub_hazard',     'B  Subsidence Hazard'),
    ('climate_stress', 'C  Climate Stress'),
    ('exposure',       'D  Exposure'),
    ('vulnerability',  'E  Vulnerability'),
    ('cgsri',          'F  CGSRI (composite)'),
]

fig, axes = plt.subplots(2, 3, figsize=(16, 10),
                          subplot_kw={'projection': ccrs.PlateCarree()})
axes = axes.flatten()
fig.suptitle('CGSRI Components and Composite Risk — Coastal Ghana',
             fontsize=12, fontweight='bold')

for ax, (col, title) in zip(axes, components):
    cmap = 'RdYlGn_r' if col == 'cgsri' else 'YlOrRd'
    sc = ax.scatter(grid['centroid_lon'], grid['centroid_lat'],
                   c=grid[col], cmap=cmap, s=12, vmin=0, vmax=1, alpha=0.85,
                   transform=ccrs.PlateCarree())
    ax.add_feature(cfeature.COASTLINE, lw=0.6)
    ax.add_feature(cfeature.LAND, facecolor='#f0f0f0', zorder=0)
    ax.set_extent([-3.5, 1.5, 4.0, 11.5], crs=ccrs.PlateCarree())
    ax.set_title(title, loc='left', fontsize=9)
    gl = ax.gridlines(draw_labels=False, lw=0.2, alpha=0.3)

cbar_ax = fig.add_axes([0.92, 0.15, 0.015, 0.7])
sm = plt.cm.ScalarMappable(cmap='YlOrRd', norm=mcolors.Normalize(0,1))
sm.set_array([])
fig.colorbar(sm, cax=cbar_ax, label='Component score (0–1)')

fig.tight_layout(rect=[0,0,0.91,0.96])
fig.savefig(FIG_DIR / 'cgsri_component_maps.png', dpi=150, bbox_inches='tight')
plt.close(fig)
print('Saved:', FIG_DIR / 'cgsri_component_maps.png')


In [ ]:
# 13. Figure: final CGSRI risk map with priority zones
import cartopy.crs as ccrs
import cartopy.feature as cfeature

risk_colours = {
    'Very low': '#1a9641', 'Low': '#a6d96a', 'Moderate': '#ffffbf',
    'High': '#fdae61', 'Very high': '#d7191c',
}
grid['cgsri_colour'] = grid['cgsri_class'].map(risk_colours).fillna('#cccccc')

fig = plt.figure(figsize=(11, 11))
ax  = fig.add_subplot(111, projection=ccrs.PlateCarree())

for cls, colour in risk_colours.items():
    subset = grid[grid['cgsri_class'] == cls]
    if len(subset) > 0:
        ax.scatter(subset['centroid_lon'], subset['centroid_lat'],
                   color=colour, s=20, alpha=0.85, label=cls,
                   transform=ccrs.PlateCarree())

# Priority zones
ax.scatter(priority['centroid_lon'], priority['centroid_lat'],
           facecolors='none', edgecolors='black', s=90, lw=1.5,
           transform=ccrs.PlateCarree(), zorder=6, label='Priority intervention')

ax.add_feature(cfeature.COASTLINE, lw=0.8)
ax.add_feature(cfeature.BORDERS, lw=0.5, linestyle=':')
ax.add_feature(cfeature.LAND, facecolor='#f5f5f5', zorder=0)
ax.add_feature(cfeature.RIVERS, edgecolor='#4292c6', lw=0.4)

cities = {'Accra':(-0.19,5.55),'Tema':(0.00,5.67),'Takoradi':(-1.75,4.90),
          'Cape Coast':(-1.25,5.12),'Kumasi':(-1.62,6.69),
          'Ho':(0.47,6.60),'Keta':(0.99,5.92)}
for city,(lon_c,lat_c) in cities.items():
    ax.plot(lon_c, lat_c, 'ks', ms=5, transform=ccrs.PlateCarree(), zorder=7)
    ax.text(lon_c+0.06, lat_c+0.06, city, fontsize=7.5,
            transform=ccrs.PlateCarree(), zorder=7)

legend_patches = [Patch(facecolor=risk_colours[cls], label=cls, edgecolor='grey')
                  for cls in risk_colours]
legend_patches.append(
    plt.Line2D([0],[0], marker='o', color='w', markerfacecolor='none',
               markeredgecolor='black', markersize=9, label='Priority zone')
)
ax.legend(handles=legend_patches, fontsize=9, loc='upper right',
          title='Risk class', title_fontsize=9)

ax.set_extent([-3.5, 1.5, 4.0, 11.5], crs=ccrs.PlateCarree())
ax.set_title(
    'Coastal Groundwater-Subsidence Risk Index (CGSRI) — Ghana\n'
    'GW hazard + Subsidence hazard + Climate stress + Exposure + Vulnerability\n'
    '(Quintile classification; circle = priority intervention zone)',
    fontsize=10
)
gl = ax.gridlines(draw_labels=True, lw=0.3, alpha=0.5)
gl.top_labels = gl.right_labels = False

fig.tight_layout()
fig.savefig(FIG_DIR / 'cgsri_final_risk_map.png', dpi=180, bbox_inches='tight')
plt.close(fig)
print('Saved:', FIG_DIR / 'cgsri_final_risk_map.png')


In [ ]:
# 14. Stage 10 summary
print('=' * 60)
print('Stage 10 complete — Integrated Risk Index')
outputs = [
    PROC_RISK / 'cgsri_grid10km.gpkg',
    PROC_RISK / 'cgsri_grid10km.csv',
    TAB_DIR / 'cgsri_district_ranking.csv',
    TAB_DIR / 'cgsri_priority_zones.csv',
    FIG_DIR / 'cgsri_component_maps.png',
    FIG_DIR / 'cgsri_final_risk_map.png',
]
for p in outputs:
    tick = 'OK' if Path(p).exists() else 'MISSING'
    print(f'  [{tick}]  {Path(p).name}')

print()
print(f'Total cells: {len(grid)}')
print(f'Priority intervention zones: {len(priority)}')
print(f'Population in priority zones: {priority["population_2020"].sum():,.0f}')
print()
print('CGSRI data sources and limitations:')
print('  1. GRACE land-mask covers eastern Ghana only (lon 0.25-1.25E)')
print('     Western Ghana cells use CHIRPS/urban proxies for GW stress')
insar_cells = int((grid['insar_n_valid'] >= 10).sum())
print(f'  2. Subsidence hazard: MintPy SBAS InSAR blended for {insar_cells} cells')
print('     ({:.1f}% of grid); proxy (elevation + density) for remaining cells'.format(
    insar_cells / len(grid) * 100))
print('     InSAR LOS ≠ vertical; atmospheric noise not fully corrected')
print('  3. Piped water access uses distance-to-city proxy, not measured access')
print('  4. Equal weights — sensitivity analysis recommended before publication')
